# GNN Supply Chain Risk — Lite Version

**Trial-account compatible** risk analysis using `networkx` + `scikit-learn`.  
No GPU, no SPCS, no External Access Integration required.  
Runs on standard Snowflake Notebooks (CPU).

## Algorithm

| Original (PyTorch Geometric) | This Notebook |
|------------------------------|---------------|
| GraphSAGE node embeddings | PageRank (personalized) |
| GNN risk propagation | PageRank + Betweenness Centrality blend |
| Link prediction | Trade data consignee name matching |
| Bottleneck detection | High fan-out external shippers |

## Output Tables
- `SF_SOLUTIONS.GNN_SUPPLY_CHAIN_RISK.RISK_SCORES`
- `SF_SOLUTIONS.GNN_SUPPLY_CHAIN_RISK.PREDICTED_LINKS`
- `SF_SOLUTIONS.GNN_SUPPLY_CHAIN_RISK.BOTTLENECKS`

In [ ]:
import warnings

warnings.filterwarnings("ignore")

from difflib import SequenceMatcher

import networkx as nx
import numpy as np
import pandas as pd
from snowflake.snowpark.context import get_active_session

session = get_active_session()

DB_SCHEMA = "SF_SOLUTIONS.GNN_SUPPLY_CHAIN_RISK"
MODEL_VERSION = "networkx-lite-v1"

print(f"Connected: {session.get_current_account()} / {session.get_current_database()}")
print(f"Model version: {MODEL_VERSION}")

In [ ]:
print("Loading data from Snowflake...")

vendors_df = session.sql(f"SELECT * FROM {DB_SCHEMA}.VENDORS").to_pandas()
materials_df = session.sql(f"SELECT * FROM {DB_SCHEMA}.MATERIALS").to_pandas()
po_df = session.sql(f"SELECT * FROM {DB_SCHEMA}.PURCHASE_ORDERS").to_pandas()
bom_df = session.sql(f"SELECT * FROM {DB_SCHEMA}.BILL_OF_MATERIALS").to_pandas()
trade_df = session.sql(f"SELECT * FROM {DB_SCHEMA}.TRADE_DATA").to_pandas()
regions_df = session.sql(f"SELECT * FROM {DB_SCHEMA}.REGIONS").to_pandas()

region_map = regions_df.set_index("REGION_CODE").to_dict("index")

print(f"  Vendors:          {len(vendors_df):>4}")
print(f"  Materials:        {len(materials_df):>4}")
print(f"  Purchase Orders:  {len(po_df):>4}")
print(f"  BOM entries:      {len(bom_df):>4}")
print(f"  Trade records:    {len(trade_df):>4}")
print(f"  Regions:          {len(regions_df):>4}")

In [ ]:
print("Building heterogeneous supply chain graph...")

G = nx.DiGraph()

# ── Vendor nodes ───────────────────────────────────────────────────────────
for _, v in vendors_df.iterrows():
    reg = region_map.get(v["COUNTRY_CODE"], {})
    base_risk = float(reg.get("BASE_RISK_SCORE", 0.3))
    geo_risk = float(reg.get("GEOPOLITICAL_RISK", 0.3))
    fin_risk = 1.0 - float(v["FINANCIAL_HEALTH_SCORE"])
    node_risk = 0.50 * fin_risk + 0.30 * base_risk + 0.20 * geo_risk
    G.add_node(
        f"V_{v['VENDOR_ID']}",
        node_type="SUPPLIER",
        raw_id=v["VENDOR_ID"],
        name=v["NAME"],
        country=v["COUNTRY_CODE"],
        initial_risk=node_risk,
    )

# ── Material nodes ─────────────────────────────────────────────────────────
for _, m in materials_df.iterrows():
    node_risk = float(1.0 - m["CRITICALITY_SCORE"]) * 0.5 + 0.1
    G.add_node(
        f"M_{m['MATERIAL_ID']}",
        node_type="PART",
        raw_id=m["MATERIAL_ID"],
        name=m["DESCRIPTION"],
        initial_risk=node_risk,
    )

# ── External shipper nodes (from trade data) ───────────────────────────────
shipper_stats = (
    trade_df.groupby("SHIPPER_NAME").agg(COUNTRY=("SHIPPER_COUNTRY", "first"), COUNT=("BOL_ID", "count")).reset_index()
)
for _, s in shipper_stats.iterrows():
    reg = region_map.get(s["COUNTRY"], {})
    node_risk = float(reg.get("BASE_RISK_SCORE", 0.4))
    G.add_node(
        f"E_{s['SHIPPER_NAME']}",
        node_type="EXTERNAL_SUPPLIER",
        raw_id=s["SHIPPER_NAME"],
        name=s["SHIPPER_NAME"],
        country=s["COUNTRY"],
        shipment_count=int(s["COUNT"]),
        initial_risk=node_risk,
    )

# ── Edges: Purchase Orders (vendor → material) ─────────────────────────────
for _, po in po_df.iterrows():
    src = f"V_{po['VENDOR_ID']}"
    dst = f"M_{po['MATERIAL_ID']}"
    spend = float(po["QUANTITY"]) * float(po["UNIT_PRICE"])
    if G.has_node(src) and G.has_node(dst):
        if G.has_edge(src, dst):
            G[src][dst]["weight"] += spend
        else:
            G.add_edge(src, dst, weight=spend, edge_type="SUPPLIES")

# ── Edges: Bill of Materials (parent → child) ──────────────────────────────
for _, b in bom_df.iterrows():
    src = f"M_{b['PARENT_MATERIAL_ID']}"
    dst = f"M_{b['CHILD_MATERIAL_ID']}"
    if G.has_node(src) and G.has_node(dst):
        G.add_edge(src, dst, weight=float(b["QUANTITY_PER_UNIT"]), edge_type="CONTAINS")

# ── Edges: Trade data (shipper → matched Tier-1 vendor) ───────────────────
vendor_names = [(v["VENDOR_ID"], v["NAME"].lower()) for _, v in vendors_df.iterrows()]
trade_edges = []  # list of (shipper_name, vendor_id, match_score)


def best_vendor_match(consignee: str, threshold: float = 0.65):
    """Find best matching vendor for a consignee name."""
    consignee_l = consignee.lower()
    best_id, best_score = None, 0.0
    for vid, vname in vendor_names:
        if vname in consignee_l or consignee_l in vname:
            return vid, 1.0
        score = SequenceMatcher(None, vname, consignee_l).ratio()
        if score > best_score:
            best_score, best_id = score, vid
    return (best_id, best_score) if best_score >= threshold else (None, 0.0)


for _, t in trade_df.iterrows():
    shipper_node = f"E_{t['SHIPPER_NAME']}"
    vid, score = best_vendor_match(t["CONSIGNEE_NAME"])
    if vid:
        vendor_node = f"V_{vid}"
        if G.has_node(shipper_node) and G.has_node(vendor_node):
            if G.has_edge(shipper_node, vendor_node):
                G[shipper_node][vendor_node]["trade_count"] += 1
            else:
                G.add_edge(
                    shipper_node,
                    vendor_node,
                    weight=1.0,
                    edge_type="SHIPS_TO",
                    trade_count=1,
                    match_score=score,
                )
            trade_edges.append((t["SHIPPER_NAME"], vid, score))

print(f"Graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")
type_counts = pd.Series(nx.get_node_attributes(G, "node_type")).value_counts()
print(type_counts.to_string())
print(f"Trade edges matched: {len(trade_edges)}")

In [ ]:
print("Computing risk scores (PageRank + Betweenness Centrality)...")

# Personalized PageRank: each node seeded with its initial risk
initial_risks = nx.get_node_attributes(G, "initial_risk")
total_risk = sum(initial_risks.values()) or 1.0
personalization = {n: v / total_risk for n, v in initial_risks.items()}

pagerank = nx.pagerank(
    G,
    alpha=0.85,
    personalization=personalization,
    weight="weight",
    max_iter=200,
    tol=1e-6,
)

# Betweenness centrality (unweighted for speed)
betweenness = nx.betweenness_centrality(G, normalized=True)


# Normalise both to [0, 1]
def norm01(vals_dict):
    """Normalize values dict to [0, 1] range."""
    arr = np.array(list(vals_dict.values()))
    lo, hi = arr.min(), arr.max()
    normed = (arr - lo) / (hi - lo + 1e-9)
    return dict(zip(vals_dict.keys(), normed))


pr_norm = norm01(pagerank)
bc_norm = norm01(betweenness)

# Compute final risk for SUPPLIER and PART nodes only
risk_records = []
for node_id, data in G.nodes(data=True):
    ntype = data["node_type"]
    if ntype not in ("SUPPLIER", "PART"):
        continue

    base = data["initial_risk"]
    pr = pr_norm.get(node_id, 0.0)
    bc = bc_norm.get(node_id, 0.0)
    score = float(np.clip(0.50 * base + 0.35 * pr + 0.15 * bc, 0.05, 0.95))

    if score >= 0.70:
        category = "CRITICAL"
    elif score >= 0.50:
        category = "HIGH"
    elif score >= 0.30:
        category = "MEDIUM"
    else:
        category = "LOW"

    confidence = float(np.clip(0.60 + 0.40 * bc, 0.55, 0.95))

    risk_records.append(
        {
            "NODE_ID": data["raw_id"],
            "NODE_TYPE": ntype,
            "RISK_SCORE": round(score, 4),
            "RISK_CATEGORY": category,
            "CONFIDENCE": round(confidence, 4),
            "MODEL_VERSION": MODEL_VERSION,
        }
    )

risk_df = pd.DataFrame(risk_records)
print(f"Risk scores computed: {len(risk_df)} nodes")
print(risk_df["RISK_CATEGORY"].value_counts().to_string())
print(f"\nAvg risk score:  {risk_df['RISK_SCORE'].mean():.3f}")
print(f"CRITICAL nodes:  {(risk_df['RISK_CATEGORY'] == 'CRITICAL').sum()}")
print(f"HIGH nodes:      {(risk_df['RISK_CATEGORY'] == 'HIGH').sum()}")

In [ ]:
print("Predicting hidden Tier-2+ links from trade data patterns...")

# Aggregate: shipper → {vendor_id: {count, max_match_score}}
shipper_vendor_map: dict = {}
for shipper_name, vendor_id, match_score in trade_edges:
    entry = shipper_vendor_map.setdefault(shipper_name, {})
    if vendor_id not in entry:
        entry[vendor_id] = {"count": 0, "match_score": match_score}
    entry[vendor_id]["count"] += 1
    entry[vendor_id]["match_score"] = max(entry[vendor_id]["match_score"], match_score)

link_records = []
for shipper_name, vendor_map in shipper_vendor_map.items():
    shipper_node = f"E_{shipper_name}"
    if not G.has_node(shipper_node):
        continue

    for vendor_id, stats in vendor_map.items():
        trade_count = stats["count"]
        match_score = stats["match_score"]

        # Probability: 0.50 base + 0.05 per shipment, capped at 0.95
        probability = min(0.50 + 0.05 * trade_count, 0.95)

        if trade_count >= 5:
            evidence = "STRONG"
        elif trade_count >= 2:
            evidence = "MODERATE"
        else:
            evidence = "WEAK"

        link_records.append(
            {
                "SOURCE_NODE_ID": shipper_name,
                "SOURCE_NODE_TYPE": "EXTERNAL_SUPPLIER",
                "TARGET_NODE_ID": vendor_id,
                "TARGET_NODE_TYPE": "SUPPLIER",
                "LINK_TYPE": "INFERRED_SUPPLIES",
                "PROBABILITY": round(probability, 4),
                "EVIDENCE_STRENGTH": evidence,
                "MODEL_VERSION": MODEL_VERSION,
            }
        )

link_df = pd.DataFrame(link_records) if link_records else pd.DataFrame()
if not link_df.empty:
    print(f"Predicted links:  {len(link_df)}")
    print(link_df["EVIDENCE_STRENGTH"].value_counts().to_string())
    print(f"\nAvg probability:  {link_df['PROBABILITY'].mean():.3f}")
    print("\nSample predictions:")
    print(
        link_df[["SOURCE_NODE_ID", "TARGET_NODE_ID", "PROBABILITY", "EVIDENCE_STRENGTH"]].head(8).to_string(index=False)
    )
else:
    print("No trade matches found — check TRADE_DATA.CONSIGNEE_NAME vs VENDORS.NAME")

In [ ]:
print("Detecting bottlenecks (single points of failure)...")

risk_lookup = dict(zip(risk_df["NODE_ID"], risk_df["RISK_SCORE"]))

bottleneck_records = []
for node_id, data in G.nodes(data=True):
    if data["node_type"] != "EXTERNAL_SUPPLIER":
        continue

    # Downstream SUPPLIER nodes this external shipper delivers to
    dependent_vendors = [
        G.nodes[s]["raw_id"] for s in G.successors(node_id) if G.nodes[s].get("node_type") == "SUPPLIER"
    ]

    if len(dependent_vendors) < 2:
        continue  # only flag shippers with 2+ Tier-1 dependents

    dep_risks = [risk_lookup.get(v, 0.3) for v in dependent_vendors]
    avg_dep_risk = float(np.mean(dep_risks))

    # Impact = concentration factor + dependent risk + betweenness
    impact_score = float(
        np.clip(
            0.15 * len(dependent_vendors) + 0.50 * avg_dep_risk + 0.35 * bc_norm.get(node_id, 0.0),
            0.10,
            0.99,
        )
    )

    description = (
        f"{len(dependent_vendors)} Tier-1 vendor(s) depend on this supplier (country: {data.get('country', 'UNK')})"
    )

    bottleneck_records.append(
        {
            "NODE_ID": data["raw_id"],
            "NODE_TYPE": "EXTERNAL_SUPPLIER",
            "DEPENDENT_COUNT": len(dependent_vendors),
            "IMPACT_SCORE": round(impact_score, 4),
            "DESCRIPTION": description,
            "MITIGATION_STATUS": "UNMITIGATED",
        }
    )

bottleneck_df = (
    pd.DataFrame(bottleneck_records).sort_values("IMPACT_SCORE", ascending=False)
    if bottleneck_records
    else pd.DataFrame()
)

print(f"Bottlenecks identified: {len(bottleneck_df)}")
if not bottleneck_df.empty:
    print("\nTop bottlenecks by impact:")
    print(bottleneck_df[["NODE_ID", "DEPENDENT_COUNT", "IMPACT_SCORE", "DESCRIPTION"]].head(5).to_string(index=False))

In [ ]:
print("Writing results to Snowflake...")

session.sql(f"TRUNCATE TABLE IF EXISTS {DB_SCHEMA}.RISK_SCORES").collect()
session.sql(f"TRUNCATE TABLE IF EXISTS {DB_SCHEMA}.PREDICTED_LINKS").collect()
session.sql(f"TRUNCATE TABLE IF EXISTS {DB_SCHEMA}.BOTTLENECKS").collect()

# ── RISK_SCORES ─────────────────────────────────────────────────────────────
(
    session.create_dataframe(risk_df)
    .select("NODE_ID", "NODE_TYPE", "RISK_SCORE", "RISK_CATEGORY", "CONFIDENCE", "MODEL_VERSION")
    .write.mode("append")
    .save_as_table(f"{DB_SCHEMA}.RISK_SCORES")
)
print(f"  RISK_SCORES:      {len(risk_df):>4} rows written")

# ── PREDICTED_LINKS ─────────────────────────────────────────────────────────
if not link_df.empty:
    (
        session.create_dataframe(link_df)
        .select(
            "SOURCE_NODE_ID",
            "SOURCE_NODE_TYPE",
            "TARGET_NODE_ID",
            "TARGET_NODE_TYPE",
            "LINK_TYPE",
            "PROBABILITY",
            "EVIDENCE_STRENGTH",
            "MODEL_VERSION",
        )
        .write.mode("append")
        .save_as_table(f"{DB_SCHEMA}.PREDICTED_LINKS")
    )
    print(f"  PREDICTED_LINKS:  {len(link_df):>4} rows written")
else:
    print("  PREDICTED_LINKS:     0 rows (no trade matches found)")

# ── BOTTLENECKS ─────────────────────────────────────────────────────────────
if not bottleneck_df.empty:
    (
        session.create_dataframe(bottleneck_df)
        .select("NODE_ID", "NODE_TYPE", "DEPENDENT_COUNT", "IMPACT_SCORE", "DESCRIPTION", "MITIGATION_STATUS")
        .write.mode("append")
        .save_as_table(f"{DB_SCHEMA}.BOTTLENECKS")
    )
    print(f"  BOTTLENECKS:      {len(bottleneck_df):>4} rows written")
else:
    print("  BOTTLENECKS:         0 rows (no multi-vendor shippers found)")

# ── Verification ─────────────────────────────────────────────────────────────
print("\n--- Table Counts ---")
verify = session.sql(f"""
    SELECT
        (SELECT COUNT(*) FROM {DB_SCHEMA}.RISK_SCORES)     AS RISK_SCORES,
        (SELECT COUNT(*) FROM {DB_SCHEMA}.PREDICTED_LINKS) AS PREDICTED_LINKS,
        (SELECT COUNT(*) FROM {DB_SCHEMA}.BOTTLENECKS)     AS BOTTLENECKS,
        (SELECT COUNT(*) FROM {DB_SCHEMA}.RISK_SCORES WHERE RISK_CATEGORY = 'CRITICAL') AS CRITICAL,
        (SELECT COUNT(*) FROM {DB_SCHEMA}.RISK_SCORES WHERE RISK_CATEGORY = 'HIGH')     AS HIGH_RISK
""").to_pandas()
print(verify.to_string(index=False))
print("\nDone! Streamlit dashboard and Cortex Agent are now ready.")